# Explore the B3 SQLite database

Use this notebook to check what was loaded by `ingest.py`.

**Before you start**
1. Run `uv sync` (once)
2. Run `uv run python ingest.py` so `data/b3.db` exists
3. Open this notebook and run the cells from top to bottom

Tip: change the SQL in the last cells to try your own queries.

## 1. Connect to the database

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

# Path to the DB created by ingest.py
DB_PATH = Path("data") / "b3.db"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH.resolve()}.\n"
        "Run: uv run python ingest.py"
    )

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH.resolve()}")

## 2. List all tables

In [ ]:
tables = pd.read_sql(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    conn,
)
tables

## 3. Row counts per table

In [ ]:
counts = []
for table_name in tables["table_name"]:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM "{table_name}"', conn)["n"].iloc[0]
    counts.append({"table_name": table_name, "rows": n})

pd.DataFrame(counts)

## 4. Peek at each table (first rows)

Change `N` if you want to see more rows.

In [ ]:
N = 5

for table_name in tables["table_name"]:
    print("=" * 80)
    print(table_name)
    display(pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT {N}', conn))

## 5. Example queries

Edit these cells and re-run them.

In [ ]:
# All stock positions for a month
pd.read_sql(
    """
    SELECT *
    FROM raw_posicao_acoes
    WHERE report_month = '2026-08'
    """,
    conn,
)

In [ ]:
# Provents (dividends / income) received
pd.read_sql(
    """
    SELECT *
    FROM raw_proventos
    ORDER BY Pagamento
    """,
    conn,
)

In [ ]:
# Negotiations / movements
pd.read_sql(
    """
    SELECT *
    FROM raw_negociacoes
    """,
    conn,
)

In [ ]:
# Your own query — replace the SQL below
pd.read_sql(
    """
    SELECT report_month, COUNT(*) AS rows
    FROM raw_posicao_fundos
    GROUP BY report_month
    """,
    conn,
)

## 6. Close the connection

Run this when you are done exploring.

In [ ]:
conn.close()
print("Connection closed.")